In [3]:
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect(':memory:')

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    conn.executescript(sql)
    conn.commit()

# Path to the 4 CSVs (downloaded from the course site)
DATA_DIR = os.path.expanduser('/content')

print('SQL engine ready.  sqlite3 version:', sqlite3.sqlite_version)
print('Data dir            :', DATA_DIR)
print('Files present       :', sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else 'NOT FOUND')

SQL engine ready.  sqlite3 version: 3.37.2
Data dir            : /content
Files present       : ['.config', 'companies.csv', 'employees.CSV', 'functions.csv', 'salaries.csv', 'sample_data']


In [4]:
# Load the four CSVs into the SQLite engine
import pandas as pd

# Note: encoding='utf-8-sig' handles the byte-order-mark (BOM) on companies.csv
# Note: decimal=',' converts European-style "6335,56" -> 6335.56 on the salary column
companies = pd.read_csv(f'{DATA_DIR}/companies.csv',
                        sep=';', encoding='utf-8-sig')
employees = pd.read_csv(f'{DATA_DIR}/employees.CSV',
                        sep=';', encoding='utf-8-sig')
functions = pd.read_csv(f'{DATA_DIR}/functions.csv',
                        sep=';', encoding='utf-8-sig')
salaries  = pd.read_csv(f'{DATA_DIR}/salaries.csv',
                        sep=';', encoding='utf-8-sig', decimal=',')

# Drop any pre-existing tables (re-run safety)
run('DROP TABLE IF EXISTS companies; DROP TABLE IF EXISTS employees; '
    'DROP TABLE IF EXISTS functions; DROP TABLE IF EXISTS salaries; '
    'DROP TABLE IF EXISTS df_employee;')

# Push DataFrames into SQLite
companies.to_sql('companies', conn, index=False)
employees.to_sql('employees', conn, index=False)
functions.to_sql('functions', conn, index=False)
salaries.to_sql('salaries',   conn, index=False)

# Sanity-check row counts
print('companies :', len(companies))
print('employees :', len(employees))
print('functions :', len(functions))
print('salaries  :', len(salaries))

companies : 25
employees : 1156
functions : 107
salaries  : 8049


🌟 Exercise 1: Building a Comprehensive Dataset for Employee Analysis


In [5]:
run('''
CREATE TABLE df_employee AS
SELECT
    s.employee_id || '_' || s.date AS id,
    DATE(SUBSTR(s.date, 7, 4) || '-' || SUBSTR(s.date, 4, 2) || '-' || SUBSTR(s.date, 1, 2)) AS month_year,
    s.employee_id,
    s.employee_name,
    e."GEN(M_F)"      AS gender,
    e.age,
    s.salary,
    f.function_group,
    c.company_name,
    c.company_city,
    c.company_state,
    c.company_type,
    c.const_site_category
FROM salaries s
LEFT JOIN employees e ON e.employee_code_emp = s.employee_id
LEFT JOIN functions f ON f.function_code     = s.func_code
LEFT JOIN companies c ON c.company_name      = s.comp_name
''')

q('SELECT * FROM df_employee LIMIT 5')

,id,month_year,employee_id,employee_name,gender,age,salary,function_group,company_name,company_city,company_state,company_type,const_site_category
0,26193_01/01/2022 00:00,2022-01-01,26193,Jacob Smith,M,38,6335.56,Managers,DM Company Head Quarters,Goiania,GOIAS,Administration,None
1,25322_01/01/2022 00:00,2022-01-01,25322,Michael Johnson,M,30,2619.35,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
2,27602_01/01/2022 00:00,2022-01-01,27602,Matthew Williams,M,28,1221.67,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
3,27127_01/01/2022 00:00,2022-01-01,27127,Joshua Brown,M,25,4000.00,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
4,23007_01/01/2022 00:00,2022-01-01,23007,Christopher Jones,M,25,3012.52,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None


🌟 Exercise 2: Cleaning Data for Consistency and Quality


In [6]:
q('SELECT * FROM df_employee LIMIT 10')

,id,month_year,employee_id,employee_name,gender,age,salary,function_group,company_name,company_city,company_state,company_type,const_site_category
0,26193_01/01/2022 00:00,2022-01-01,26193,Jacob Smith,M,38,6335.56,Managers,DM Company Head Quarters,Goiania,GOIAS,Administration,None
1,25322_01/01/2022 00:00,2022-01-01,25322,Michael Johnson,M,30,2619.35,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
2,27602_01/01/2022 00:00,2022-01-01,27602,Matthew Williams,M,28,1221.67,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
3,27127_01/01/2022 00:00,2022-01-01,27127,Joshua Brown,M,25,4000.00,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
4,23007_01/01/2022 00:00,2022-01-01,23007,Christopher Jones,M,25,3012.52,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
5,25634_01/01/2022 00:00,2022-01-01,25634,Nicholas Miller,M,32,1549.28,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
6,23198_01/01/2022 00:00,2022-01-01,23198,Andrew Davis,M,18,3000.00,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
7,18876_01/01/2022 00:00,2022-01-01,18876,Joseph Garcia,M,24,3266.67,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
8,26830_01/01/2022 00:00,2022-01-01,26830,Daniel Rodriguez,M,22,3000.00,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
9,28590_01/01/2022 00:00,2022-01-01,28590,William Wilson,M,19,2500.00,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None


In [13]:
run('''
    UPDATE df_employee
    SET
    id = TRIM(id),
    company_name = TRIM(company_name),
    employee_name = TRIM(employee_name),
    month_year = TRIM(month_year),
    function_group = TRIM(function_group),
    salary = TRIM(salary)''')

In [14]:
q('''
  SELECT * FROM df_employee
  WHERE id IS NULL
   OR month_year IS NULL
   OR employee_id IS NULL
   OR salary IS NULL
   OR salary = ' '
   OR salary = ''
   ''')

,id,month_year,employee_id,employee_name,gender,age,salary,function_group,company_name,company_city,company_state,company_type,const_site_category
0,29837_01/01/2022 00:00,2022-01-01,29837,Jason Sanchez,M,19,None,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
1,30395_01/01/2022 00:00,2022-01-01,30395,Deon Greer,M,19,None,Administration,The Oasis at Desert Springs,Palmas,Tocantins,Construction Site,Residential
2,28767_01/02/2022 00:00,2022-02-01,28767,Thomas Perez,M,21,None,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
3,29836_01/02/2022 00:00,2022-02-01,29836,Jose Hall,M,19,None,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
4,29837_01/02/2022 00:00,2022-02-01,29837,Jason Sanchez,M,19,None,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,30754_01/01/2023 00:00,2023-01-01,30754,Izaiah Bond,M,27,None,Professionals,The Glades at Maplewood,Goiania,GOIAS,Construction Site,Commercial
66,24877_01/01/2023 00:00,2023-01-01,24877,Zackary Matthews,M,28,None,Assistants,The Glades at Maplewood,Goiania,GOIAS,Construction Site,Commercial
67,32729_01/01/2023 00:00,2023-01-01,32729,Julie Henderson,F,23,None,Assistants,The Glades at Maplewood,Goiania,GOIAS,Construction Site,Commercial
68,30111_01/01/2023 00:00,2023-01-01,30111,Arturo Fuller,M,25,None,Assistants,The Glades at Maplewood,Goiania,GOIAS,Construction Site,Commercial


In [16]:
run('''
DELETE FROM df_employee
WHERE salary = ' '
   OR salary IS NULL
   OR salary = '';
''')

🌟 Exercise 3 : Calculating Current Employee Counts by Company

In [21]:
q('''
SELECT
    company_name AS company,
    COUNT(DISTINCT employee_id) AS current_employee_count
FROM df_employee
WHERE month_year = (SELECT MAX(month_year) FROM df_employee)
GROUP BY company_name
ORDER BY current_employee_count DESC
''')

,company,current_employee_count
0,The Crossings at Falcon Point,107
1,Regional Hospital,82
2,The Terraces at Diamond Heights,75
3,The Glades at Maplewood,60
4,The Pines at Windward,57
5,The Greens at Fairway Hills,57
6,The Sanctuary at Briarcliff,43
7,The Oasis at Desert Springs,39
8,The Meadows at Sunset Ridge,39
9,DM Company Head Quarters,35


🌟 Exercise 4 : Analyzing Employee Distribution by City and Over Time


In [22]:
q('''
WITH CityCounts AS (
    SELECT c.company_city, COUNT(DISTINCT s.employee_id) as emp_count
    FROM salaries s
    JOIN companies c ON s.comp_name = c.company_name
    GROUP BY c.company_city
)
SELECT company_city, emp_count,
       ROUND(100.0 * emp_count / (SELECT SUM(emp_count) FROM CityCounts), 2) as percentage
FROM CityCounts
''')

,company_city,emp_count,percentage
0,Brasilia,371,32.01
1,Goiania,734,63.33
2,Goianiaa,2,0.17
3,Palmas,52,4.49


🌟 Exercise 5 : Analyzing Employment Trends and Salary Metrics



In [23]:
q('''
WITH MonthlyCounts AS (
    SELECT date, COUNT(DISTINCT employee_id) as emp_count
    FROM salaries GROUP BY date
)
SELECT * FROM MonthlyCounts
WHERE emp_count = (SELECT MIN(emp_count) FROM MonthlyCounts)
OR emp_count = (SELECT MAX(emp_count) FROM MonthlyCounts)
''')

,date,emp_count
0,01/01/2022 00:00,520
1,01/01/2023 00:00,676
